<a href="https://colab.research.google.com/github/lucaspaludo/15-pulse-counter/blob/main/6_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

import matplotlib.pyplot as plt
import torch
import numpy as np

import matplotlib.pyplot as plt
import torch
import numpy as np

In [36]:

# convertendo MNIST para um tensor de 4 dimensões (nº imagens, largura, altura, cor do canal)

transform = transforms.ToTensor()

In [37]:
# Train data
train_data = datasets.MNIST(root='/cnn_data', train=True, download=True, transform=transform)

In [38]:

# Test data
test_data = datasets.MNIST(root='/cnn_data', train=False, download=True, transform=transform)

In [39]:
class ConvolutionalNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 6, 3, 1)
    self.conv2 = nn.Conv2d(6, 16, 3, 1)

    # camadas totalmente conectadas
    self.fc1 = nn.Linear(5*5*16, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, 10)

  def  forward(self, X):
    X = F.relu(self.conv1(X))
    X = F.max_pool2d(X, 2, 2)

    # segunda passada
    X = F.relu(self.conv2(X))
    X = F.max_pool2d(X, 2, 2)

    # redimensionar para flatten
    X = X.view(-1, 16*5*5) # -1 para podermos variar o tamanho do lote

    # camadas totalmente conectadas
    X = F.relu(self.fc1(X))
    X = F.relu(self.fc2(X))
    X = self.fc3(X)

    return F.log_softmax(X, dim=1)




In [40]:
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: /cnn_data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [41]:
# criando instância do modelo
torch.manual_seed(42)
model = ConvolutionalNetwork()
model

ConvolutionalNetwork(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [42]:
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: /cnn_data
    Split: Test
    StandardTransform
Transform: ToTensor()

In [43]:
# criando um batch de 10

train_loader = DataLoader(train_data, batch_size=10, shuffle=True)
test_loader = DataLoader(test_data, batch_size=10, shuffle=False)

In [44]:
# função de otimização de perda
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [45]:
import time
start_time = time.time()

# criando variáveis
epochs = 5
train_losses = []
test_losses = []
train_correct = []
test_correct = []

# loop de épocas
for i in range(epochs):
  trn_corr = 0
  tst_corr = 0

  # treino
  for b, (X_train, y_train) in enumerate(train_loader):
    b += 1 # iniciando batches em 1
    y_pred = model(X_train) # obtém previsões para o dataset de treino
    loss = criterion(y_pred, y_train) # calcular perda, compara previsões com as respostas corretas

    predicted = torch.max(y_pred.data, 1)[1] # incrementando número de previsões corretas
    batch_corr = (predicted == y_train).sum() # quantos acertos temos nesse batch específico 1 -> True, 0 -> False
    trn_corr += batch_corr # acompanhe enquanto treinamos sozinhos

    # atualizando parâmetros
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # printando os resultados
    if b % 600 == 0:
      print(f"Epoch: {i}  Batch: {b}  Loss: {loss.item()}")

  train_losses.append(loss)
  train_correct.append(trn_corr)

  # teste
  with torch.no_grad(): # sem gradiente, então não atualizamos os pesos para os dados de teste
    for b, (X_test, y_test) in enumerate(test_loader):
      y_val = model(X_test)
      predicted = torch.max(y_val.data, 1)[1] # incrementando previsões de teste corretas
      ts_corr = (predicted == y_test).sum() # T -> 1,  F -> 0 e some tudo

  loss = criterion(y_val, y_test)
  test_losses.append(loss)
  test_correct.append(tst_corr)

current_time = time.time()
total = current_time - start_time
print(f"O treinmaneot levou: {total/60} minutos")


Epoch: 0  Batch: 600  Loss: 0.04732787609100342
Epoch: 0  Batch: 1200  Loss: 0.07334671914577484
Epoch: 0  Batch: 1800  Loss: 0.3151721656322479
Epoch: 0  Batch: 2400  Loss: 0.026600707322359085
Epoch: 0  Batch: 3000  Loss: 0.005466829054057598
Epoch: 0  Batch: 3600  Loss: 0.0017479986418038607
Epoch: 0  Batch: 4200  Loss: 0.5357739925384521
Epoch: 0  Batch: 4800  Loss: 0.0563945546746254
Epoch: 0  Batch: 5400  Loss: 0.006266170646995306
Epoch: 0  Batch: 6000  Loss: 0.02795628271996975
Epoch: 1  Batch: 600  Loss: 0.04767467826604843
Epoch: 1  Batch: 1200  Loss: 0.040824078023433685
Epoch: 1  Batch: 1800  Loss: 0.00163908745162189
Epoch: 1  Batch: 2400  Loss: 0.0879412293434143
Epoch: 1  Batch: 3000  Loss: 0.32062822580337524
Epoch: 1  Batch: 3600  Loss: 0.00038606187445111573
Epoch: 1  Batch: 4200  Loss: 0.0009691218147054315
Epoch: 1  Batch: 4800  Loss: 0.00044831159175373614
Epoch: 1  Batch: 5400  Loss: 0.002254956169053912
Epoch: 1  Batch: 6000  Loss: 0.0068124206736683846
Epoch: 2 